# Nifty50 Multimodal Transformer — Demo

A multimodal Transformer that fuses tabular OHLCV features, real financial news (FinBERT), GAF/MTF time-series images (CNN), and a sector/peer knowledge graph to predict short-horizon outperformance vs the Nifty50 index. The pipeline is leakage-safe with mechanically enforced integration tests and walk-forward purged cross-validation. Headline result: image (GAF/MTF) is the strongest single auxiliary modality (+2.8 pp AUC over tabular-only); signal is detectable in stable market regimes and degrades on volatility shifts.

## What this notebook covers

- **Modalities**: What each of the four modalities encodes, why it carries signal, and why it is leakage-safe.
- **Architecture**: How the Transformer fuses them.
- **Cross-validation**: How walk-forward purged CV surfaces regime-shift effects.
- **Results**: Ablation AUCs (Run C, 3-fold CV) and a corrected backtest.
- **Interpretability**: Attention-based modality attribution for individual predictions.

All cells run on pre-committed cached data — no yfinance downloads, no FinBERT model loading, no live training. Expected end-to-end runtime: under 5 minutes on a fresh Colab environment (most of that is `pip install`).

## Setup

Installs project dependencies and loads all cached data. No internet access is required after install.

In [ ]:
%%capture setup_log
import os, sys
from pathlib import Path

REPO_URL  = "https://github.com/Randhir123/nifty50-multimodal-transformer.git"
REPO_NAME = "nifty50-multimodal-transformer"

try:
    from google.colab import drive as _colab_check
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_root = Path(f"/content/{REPO_NAME}")
    if not repo_root.exists():
        ret = os.system(f"git clone {REPO_URL} {repo_root}")
        if ret != 0:
            raise RuntimeError("git clone failed — check repo URL and network")
    os.chdir(repo_root)
    os.system("pip install -r requirements.txt -q")
    os.system("pip install -e . -q")
else:
    repo_root = Path.cwd()

sys.path.insert(0, str(repo_root))
demo_data = repo_root / "notebooks" / "colab" / "demo_data"
assert demo_data.exists(), f"Demo data not found at {demo_data}"
print(f"Repo root : {repo_root}")
print(f"Demo data : {demo_data}")


In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ── modality colour palette (consistent across all plots) ──────────────────
COLORS = {"tabular": "#2c7bb6", "image": "#d7191c", "text": "#1a9641", "kg": "#fdae61"}

# ── cached data ────────────────────────────────────────────────────────────
data          = np.load(demo_data / "multimodal_samples_demo.npz", allow_pickle=False)
predictions   = pd.read_csv(demo_data / "predictions_demo.csv")
curves        = pd.read_csv(demo_data / "training_curves_demo.csv")

with open(demo_data / "run_c_summary.json") as fh:
    summary = json.load(fh)
with open(demo_data / "news_samples_demo.json") as fh:
    news_samples = json.load(fh)
with open(demo_data / "sector_mapping_demo.json") as fh:
    sector_mapping = json.load(fh)

end_dates = pd.to_datetime(data["end_dates"])
stock_ids  = np.asarray(data["stock_ids"], dtype=str)

print(f"Samples       : {len(data['y'])}")
print(f"Tabular shape : {data['tabular_tokens'].shape}  (N, window=20, features=11)")
print(f"Image shape   : {data['image_tokens'].shape}   (N, CNN_dim=16)")
print(f"Text shape    : {data['text_tokens'].shape} (N, FinBERT_dim=768)")
print(f"KG shape      : {data['kg_tokens'].shape}    (N, kg_features=4)")
print(f"Date range    : {end_dates.min().date()} → {end_dates.max().date()}")
print(f"Positive rate : {data['y'].mean():.1%}  (outperforms Nifty50 over next 3 days)")


## The four modalities

### Tabular

**What it encodes**: 11 OHLCV-derived technical features over a 20-day rolling window, covering 1-day through 10-day log returns, two realized-volatility measures, high-low range, two moving-average deviations, volume relative to 20-day average, and the stock's return minus the Nifty50 return.

**Why it carries signal**: Price momentum and relative strength have well-documented predictive power in short-horizon equity forecasting. The 20-day window gives the model a shape signature rather than just the most recent data point.

**Leakage safety**: All 11 features are computed using only data available at or before the prediction date. A manual audit confirmed no future information is used; the `test_no_leakage` integration test mechanically enforces this on every push.

In [ ]:
FEATURE_NAMES = [
    "log_return_1d", "cum_return_3d", "cum_return_5d", "cum_return_10d",
    "realized_vol_5d", "realized_vol_10d", "high_low_range_over_close",
    "close_over_10dma_minus_1", "close_over_20dma_minus_1",
    "volume_over_20d_avg", "stock_minus_index_return",
]

# Pick one RELIANCE sample near the middle of the date range
reliance_idx = np.where(stock_ids == "RELIANCE.NS")[0]
sample_idx   = reliance_idx[len(reliance_idx) // 2]
window       = data["tabular_tokens"][sample_idx]          # shape (20, 11)
sample_date  = end_dates[sample_idx].date()

tab_df = pd.DataFrame(window, columns=FEATURE_NAMES)
print(f"Sample: RELIANCE.NS, window ending {sample_date}")
print(tab_df[["log_return_1d", "cum_return_5d", "realized_vol_5d",
              "close_over_20dma_minus_1", "stock_minus_index_return"]].round(4).to_string())

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(8, 5), sharex=True)
ax1.plot(tab_df["log_return_1d"],        color=COLORS["tabular"], linewidth=1.5)
ax1.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax1.set_ylabel("log_return_1d", fontsize=8)

ax2.plot(tab_df["close_over_20dma_minus_1"], color=COLORS["tabular"], linewidth=1.5)
ax2.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax2.set_ylabel("vs 20-day MA", fontsize=8)

ax3.plot(tab_df["realized_vol_5d"],  color="#777", linewidth=1.5, label="vol 5d")
ax3.plot(tab_df["realized_vol_10d"], color="#aaa", linewidth=1.5, linestyle="--", label="vol 10d")
ax3.set_xlabel("Day in window (0=oldest, 19=newest)", fontsize=8)
ax3.set_ylabel("Realized vol", fontsize=8)
ax3.legend(fontsize=7)

fig.suptitle(f"RELIANCE.NS — 20-day tabular window ending {sample_date}", fontsize=10)
plt.tight_layout()
plt.show()


*Three of the 11 features for one RELIANCE window. The model sees all 11 features across all 20 days — the full 20×11 tensor is one token sequence fed to the tabular projection.*

### Text

**What it encodes**: A 768-dimensional FinBERT embedding of recent news and market summaries for the stock, filtered to events at or before the prediction date. FinBERT is a BERT model fine-tuned on financial text (ProsusAI/finbert). The top-5 most recent records are concatenated and encoded together.

**Why it carries signal**: News sentiment moves short-horizon returns; encoding with a financial-domain model captures richer signal than bag-of-words approaches. Distance correlation between text and tabular embeddings is 0.170 — text is independent of price-derived features, not just a re-encoding of OHLCV.

**Leakage safety**: Only records with `event_date ≤ prediction_date` are visible to a sample. Filtering is applied per sample during artifact construction.

In [ ]:
# Show a few text records for RELIANCE
ticker = "RELIANCE.NS"
print(f"Text records for {ticker} (3 most recent):\n")
for rec in news_samples.get(ticker, [])[:3]:
    print(f"  [{rec['event_date'][:10]}]  {rec['source_type']}")
    print(f"  {rec['title']}")
    print()

# FinBERT embedding distribution across the demo samples
text_emb = data["text_tokens"]              # shape (N, 768)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))

ax1.hist(text_emb[sample_idx], bins=50, color=COLORS["text"], edgecolor="none", alpha=0.85)
ax1.set_xlabel("Embedding value")
ax1.set_ylabel("Count")
ax1.set_title("Single-sample FinBERT vector\n(768 dimensions)", fontsize=9)

norms = np.linalg.norm(text_emb, axis=1)
ax2.hist(norms, bins=20, color=COLORS["text"], edgecolor="none", alpha=0.85)
ax2.set_xlabel("L2 norm")
ax2.set_title("FinBERT vector norm\nacross all demo samples", fontsize=9)

plt.tight_layout()
plt.show()


*Left: value distribution of the 768-dimensional FinBERT embedding for one sample — the smooth distribution confirms FinBERT is producing real dense embeddings, not all-zero fallbacks. Right: the L2-norm distribution is stable across samples, indicating consistent encoding quality.*

### Image (GAF/MTF + CNN)

**What it encodes**: A 2-channel image of the 20-day close-price window. Channel 1 is the Gramian Angular Field (GAF): encodes temporal correlations as angular relationships on the unit circle. Channel 2 is the Markov Transition Field (MTF): encodes transition probabilities between quantized price bins. Both are 32×32 matrices. A 3-layer CNN maps the 2×32×32 image to a 16-dimensional embedding.

**Why it carries signal**: GAF/MTF preserve temporal ordering and capture pattern shapes (momentum reversals, trending periods) that rolling statistics smooth over. Distance correlation between image and tabular embeddings is 0.047 — the CNN sees different structure than the tabular branch.

**Leakage safety**: The image is built from close prices in the 20-day window ending on the prediction date. No future prices are used; the image file is keyed by `{ticker}_{date}.npy`.

In [ ]:
from src.data.timeseries_images import compute_gaf, compute_mtf

# Load RELIANCE close prices from demo data
close_df = pd.read_csv(demo_data / "RELIANCE_NS_close_demo.csv", parse_dates=["date"])
close_df = close_df.sort_values("date").reset_index(drop=True)

# Find the window ending on sample_date
mask  = close_df["date"].dt.date <= sample_date
close = close_df.loc[mask, "close"].values[-20:]   # last 20 trading days

gaf_img = compute_gaf(close, image_size=32)
mtf_img = compute_mtf(close, image_size=32)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))

axes[0].plot(close, color=COLORS["image"], linewidth=1.5)
axes[0].set_title(f"Close price\n(20-day window → {sample_date})", fontsize=9)
axes[0].set_xlabel("Day")
axes[0].set_ylabel("Price (INR)")

im1 = axes[1].imshow(gaf_img, cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
axes[1].set_title("Gramian Angular Field\n(temporal correlations)", fontsize=9)
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(mtf_img, cmap="viridis", vmin=0, vmax=1, aspect="auto")
axes[2].set_title("Markov Transition Field\n(price-bin transitions)", fontsize=9)
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()
print(f"CNN input: 2×32×32 → CNN → 16-dim embedding")
print(f"GAF range : [{gaf_img.min():.3f}, {gaf_img.max():.3f}]")
print(f"MTF range : [{mtf_img.min():.3f}, {mtf_img.max():.3f}]")


*Left: the raw close-price series. Centre: the GAF image — red cells indicate positive angular correlation (trending), blue indicate anti-correlation (reversal). Right: the MTF image — brighter cells mean higher transition probability between the corresponding quantile bins. These two channels encode complementary aspects of temporal structure.*

### Knowledge Graph (KG)

**What it encodes**: A 4-dimensional vector of sector and peer context: peer count in sector, mean recent return of sector peers, mean recent return of sector index, and a high-volume event flag. Aligned by `(stock_id, prediction_date)`.

**Why it carries signal**: Sector rotation and peer co-movement provide context that pure price features for a single stock cannot encode. Distance correlation between KG and tabular embeddings is 0.138 — independent enough to be additive, though small.

**Leakage safety**: Peer returns are restricted to the 5-day window ending on the prediction date (`as_of_date = prediction_date`). No future peer prices are used.

In [ ]:
KG_FEATURE_NAMES = ["peer_count", "peer_avg_recent_return", "sector_avg_recent_return", "high_vol_event"]

# Show KG features for the demo samples, grouped by sector
kg_df = pd.DataFrame(data["kg_tokens"], columns=KG_FEATURE_NAMES)
kg_df["stock_id"] = stock_ids
kg_df["sector"]   = kg_df["stock_id"].map(sector_mapping["tickers"])

print("KG features — mean by sector (over demo samples):")
print(kg_df.groupby("sector")[KG_FEATURE_NAMES].mean().round(4).to_string())

print()

# Show a few individual samples
print("\nKG features — 5 sample rows:")
print(kg_df[["stock_id", "sector"] + KG_FEATURE_NAMES].head().to_string(index=False))

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for i, feat in enumerate(KG_FEATURE_NAMES):
    for sector in kg_df["sector"].unique():
        vals = kg_df.loc[kg_df["sector"] == sector, feat].values
        axes[i].hist(vals, bins=15, alpha=0.6, label=sector, edgecolor="none")
    axes[i].set_title(feat, fontsize=8)
    axes[i].set_xlabel("Value", fontsize=7)
    if i == 0:
        axes[i].legend(fontsize=6)
plt.tight_layout()
plt.show()


*Per-feature distributions split by sector. Banking stocks cluster together on peer returns; IT and energy sectors show different dispersion patterns. This sector structure is what the KG modality is meant to encode.*

## Architecture

Each modality is projected to a common embedding dimension (model_dim), a modality-type embedding is added, and all tokens are concatenated into a single sequence. One Transformer encoder layer (multi-head self-attention + feedforward) mixes information across modalities. Mean pooling over all tokens produces the final representation, which is passed to a linear classification head.

```
tabular (20 tokens × 11 dim)  → Linear(11, 16) + modality_emb[0]  ─┐
image   ( 1 token  × 16 dim)  → Linear(16, 16) + modality_emb[1]  ─┤
text    ( 1 token  × 768 dim) → Linear(768,16) + modality_emb[2]  ─┤→ concat (23, 16)
kg      ( 1 token  × 4 dim)   → Linear( 4, 16) + modality_emb[3]  ─┘
                                        ↓
                              TransformerEncoder (1 layer, 4 heads, ff_dim=32)
                                        ↓
                              mean_pool → Linear(16, 1) → logit
```

Design notes: model_dim=16 was chosen to be small enough to train on CPU in minutes on 1,000 samples. A larger model on a bigger universe would need significantly more data. Mean pooling replaced CLS-token pooling after a collapse issue was discovered: the CLS token in a 16-dim encoder consistently converged to a constant output; switching to mean pooling broke the saddle point.

In [ ]:
from src.models.fusion import FusionTransformer, FusionTransformerConfig

ckpt_path = demo_data / "model_checkpoint.pt"
ckpt      = torch.load(ckpt_path, map_location="cpu", weights_only=False)
cfg       = ckpt["config"]
model     = FusionTransformer(FusionTransformerConfig(**cfg))
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print("Model configuration:")
for k, v in cfg.items():
    print(f"  {k:22s}: {v}")

print("\nParameter counts by component:")
comp_params = {}
for name, param in model.named_parameters():
    top = name.split(".")[0]
    comp_params[top] = comp_params.get(top, 0) + param.numel()
for comp, n in sorted(comp_params.items(), key=lambda x: -x[1]):
    print(f"  {comp:25s}: {n:6,d} params")
total = sum(p.numel() for p in model.parameters())
print(f"  {'TOTAL':25s}: {total:6,d} params")


## Cross-validation and leakage safety

Walk-forward expanding-window CV: fold k trains on the oldest k portions of the data and validates on the next contiguous block. A purge gap removes any samples whose label window overlaps the train/val boundary. An embargo gap (3 days) prevents models from partially seeing near-future data through autocorrelation.

```
Fold 0:  [──── train (300) ────][purge][── val (311) ──]
Fold 1:  [──────── train (612) ────────][purge][── val (311) ──]
Fold 2:  [────────────── train (924) ──────────────][purge][── val (310) ──]
                                                  ↑ embargo gap ↑
```

Leakage safety is enforced mechanically: the integration test `tests/integration/test_no_leakage.py` fails the CI pipeline if any tabular window, text record, image file, or KG feature uses data past the sample's `end_date`.

In [ ]:
# Show the fold date ranges from the run_c_summary
fold_info = summary["fold_aucs"]
print("Run C fold breakdown (tabular_image_text_kg variant):")
print(f"  Fold 0 — {fold_info['fold_0_period']:24s}  val AUC = {fold_info['fold_0_val_auc']:.3f}")
print(f"  Fold 1 — {fold_info['fold_1_period']:24s}  val AUC = {fold_info['fold_1_val_auc']:.3f}")
print(f"  Fold 2 — {fold_info['fold_2_period']:24s}  val AUC = {fold_info['fold_2_val_auc']:.3f}")

print("\nFold 1 is the only fold with clear signal (val AUC 0.591).")
print("Fold 0 fails: label base-rate shifts by ~10 pp between train and val.")
print("Fold 2 fails: Nifty50 volatility in the val window is 2.4× the training period.")
print("Walk-forward CV surfaced this finding; a single train/test split would have hidden it.")


## Training

Training uses BCEWithLogitsLoss with class-rebalancing via `pos_weight = n_neg / n_pos`. The output bias is not manually initialized; the optimizer finds the appropriate intercept. AdamW with lr=1e-3, weight_decay=1e-4, 20 epochs, batch_size=16.

The training curves below are from a 20-epoch run on the 150-sample demo subset (not the full 1,242-sample Run C). They are illustrative of training dynamics: fast initial descent, noisy val AUC on a small dataset, no systematic overfit at this model size.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 4), sharex=True)

ax1.plot(curves["epoch"], curves["train_loss"], color="#555", linewidth=1.5, label="train")
ax1.plot(curves["epoch"], curves["val_loss"],   color="#555", linewidth=1.5, linestyle="--", label="val")
ax1.set_ylabel("BCE Loss")
ax1.legend(fontsize=8)
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

ax2.plot(curves["epoch"], curves["val_auc"], color=COLORS["tabular"], linewidth=1.5)
ax2.axhline(0.5, color="gray", linewidth=0.8, linestyle="--", label="random")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Val AUC")
ax2.legend(fontsize=8)

fig.suptitle("Training dynamics — demo subset (150 samples, 20 epochs)", fontsize=10)
plt.tight_layout()
plt.show()

best_epoch = int(curves.loc[curves["val_auc"].idxmax(), "epoch"])
best_auc   = curves["val_auc"].max()
print(f"Best val AUC = {best_auc:.4f} at epoch {best_epoch}")
print(f"(Run C full-dataset fold 1 val AUC = {summary['fold_aucs']['fold_1_val_auc']:.3f})")


## Results

Run C: 6-stock training universe (RELIANCE, TCS, INFY, HDFCBANK, ICICIBANK, SBIN), 1,242 samples, May 2025–May 2026, 3-fold purged walk-forward CV, 20 epochs. All numbers below are from the committed Run C artifact.

In [ ]:
# Ablation table
abl_rows = summary["ablation_results"]
abl_df   = pd.DataFrame(abl_rows)
abl_df["Δ AUC"] = abl_df["delta_vs_tabular"].map(lambda x: f"+{x:.4f}" if x > 0 else f"{x:.4f}")
abl_df["Mean AUC"] = abl_df["mean_auc"].map(f"{:.4f}".format)
display_df = abl_df.rename(columns={"variant": "Modality combination"})[
    ["Modality combination", "Mean AUC", "Δ AUC"]
]
print(display_df.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(7, 3.5))
variant_colors = [COLORS["tabular"], COLORS["kg"], COLORS["text"], COLORS["image"], "#9467bd"]
bars = ax.barh(display_df["Modality combination"], abl_df["mean_auc"],
               color=variant_colors, edgecolor="white", height=0.6)
ax.axvline(0.5, color="gray", linewidth=0.8, linestyle="--", label="AUC 0.5 (random)")
ax.set_xlabel("Mean Val AUC (3-fold CV)")
ax.set_title("Modality ablation — Run C", fontsize=10)
ax.set_xlim(0.47, 0.54)
for bar, row in zip(bars, abl_rows):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f"{row['mean_auc']:.4f}", va="center", fontsize=8)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


Image (GAF/MTF + CNN) is the strongest single auxiliary modality at +2.8 pp over tabular-only, more than doubling the text contribution (+1.4 pp). The KG contribution (+0.1 pp) is indistinguishable from noise at this dataset scale. The all-four combination (0.5222) is marginally below image-alone (0.5242), consistent with mild noise from text and KG on a 1,242-sample dataset. With only 3 folds and a single random seed, the ordering is interpretable but individual deltas are not precise estimates — multi-seed evaluation would be needed for confidence intervals.

In [ ]:
# Backtest equity curve (6-stock universe, corrected with real forward returns)
bt = summary["backtest_6stock"]

# Reconstruct a daily equity curve from the predictions_demo subset
# (The full Run C backtest used all 49 rebalance dates; here we illustrate with the demo predictions.)
preds_sorted = predictions.dropna(subset=["future_return_3d"]).sort_values("end_date").copy()

if len(preds_sorted) > 0:
    # Simulate: each day, pick the stock with highest predicted_score, hold for 1 day
    model_eq   = [1.0]
    bench_eq   = [1.0]
    raw_dir    = repo_root / "data/processed/real_world_demo_full/raw"
    nsei_path  = raw_dir / "NSEI.csv"

    # Aggregate predictions by date, pick top-1 stock per date
    for date_str, group in preds_sorted.groupby("end_date"):
        top_row  = group.sort_values("predicted_score", ascending=False).iloc[0]
        stock_rt = float(top_row["future_return_3d"])
        model_eq.append(model_eq[-1] * (1 + stock_rt))
        bench_rt = group["future_return_3d"].mean()   # equal-weight benchmark approx
        bench_eq.append(bench_eq[-1] * (1 + bench_rt))

    dates_plot = range(len(model_eq))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(dates_plot, model_eq, color=COLORS["tabular"],  linewidth=1.8, label="Model (top-1)")
    ax.plot(dates_plot, bench_eq, color="#aaa", linewidth=1.5, linestyle="--", label="Equal-weight benchmark")
    ax.axhline(1.0, color="gray", linewidth=0.5)
    ax.set_xlabel("Rebalance step")
    ax.set_ylabel("Portfolio value (start = 1.0)")
    ax.set_title("Backtest equity curve — demo predictions subset", fontsize=10)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("No future_return_3d data available for backtest curve.")

print("\nFull Run C backtest (6-stock, real forward returns):")
print(f"  Model total return     : {bt['model_total_return']:+.1%}")
print(f"  Benchmark total return : {bt['benchmark_total_return']:+.1%}")
print(f"  Trading days           : {bt['trading_days']}")
print(f"  Sharpe ratio           : {bt['sharpe_ratio']:.2f}")
print(f"  Max drawdown           : {bt['max_drawdown']:+.1%}")
print("\nThe model underperforms the benchmark. This is mechanically consistent with")
print("fold 2 val AUC 0.448: the top-1 selection is anti-predictive in the high-")
print("volatility Feb–May 2026 period, compounding over 49 rebalances.")


## Per-prediction interpretability

The model uses mean pooling over all 23 tokens (20 tabular + 1 image + 1 text + 1 kg). Attention weights quantify how each token's representation was formed from the full sequence — they do not directly measure prediction contribution, but they show which modality's tokens the model draws information from.

**How to read the plots**: We compute two views of the attention matrix (shape N × 23 × 23, after averaging over heads):

1. **Aggregate** (bar chart): what fraction of total attention goes to each modality. With uniform attention, tabular would receive 20/23 ≈ 87%, image/text/kg ≈ 4.3% each. Departures from this baseline reveal where the model concentrates its attention.

2. **Per-prediction** (2×2 grid): attention fraction for four representative samples — a correctly classified positive, correctly classified negative, false positive, and false negative. Each bar shows what fraction of the total attention budget was spent on that modality for that specific prediction.

In [ ]:
from src.models.fusion import FusionTransformer, FusionTransformerConfig

# Token sequence layout: 20 tabular, 1 image, 1 text, 1 kg  (total 23)
MODALITY_SLICES  = {"tabular": slice(0, 20), "image": slice(20, 21),
                    "text": slice(21, 22),   "kg":    slice(22, 23)}
MODALITY_NTOKENS = {"tabular": 20, "image": 1, "text": 1, "kg": 1}
N_TOKENS = 23

# Reconstruct model
model2 = FusionTransformer(FusionTransformerConfig(**cfg))
model2.load_state_dict(ckpt["model_state_dict"])
model2.eval()

# Hook: capture attention weights by re-running self_attn with need_weights=True
_captured = {}
def _attn_hook(module, input, output):
    q = input[0]    # [B, seq, d_model] — positional arg to MultiheadAttention
    with torch.no_grad():
        _, w = module.forward(q, q, q, need_weights=True, average_attn_weights=True)
    if w is not None:
        _captured["attn_weights"] = w.detach().cpu().numpy()  # [B, seq, seq]

handle = model2.encoder.layers[0].self_attn.register_forward_hook(_attn_hook)

tabular_t = torch.from_numpy(data["tabular_tokens"].astype(np.float32))
image_t   = torch.from_numpy(data["image_tokens"].astype(np.float32))
text_t    = torch.from_numpy(data["text_tokens"].astype(np.float32))
kg_t      = torch.from_numpy(data["kg_tokens"].astype(np.float32))

with torch.no_grad():
    logits = model2(tabular_tokens=tabular_t, image_tokens=image_t,
                    text_tokens=text_t, kg_tokens=kg_t)
    scores = torch.sigmoid(logits).numpy()

handle.remove()
attn = _captured.get("attn_weights")   # [N, 23, 23] or None

# ── aggregate attention fraction ──────────────────────────────────────────────
if attn is not None:
    print(f"Attention weights captured — shape {attn.shape}  (N, 23 queries, 23 keys)")
    print()

    # Total attention fraction: sum of attention flowing to modality / total
    # Uniform baseline: tabular=20/23≈87%, image/text/kg=1/23≈4.3% each
    total_attn = attn.sum(axis=(1, 2), keepdims=True)   # [N,1,1]  = B × 23 per sample
    modality_frac = {}
    for name, sl in MODALITY_SLICES.items():
        modality_frac[name] = float((attn[:, :, sl].sum(axis=(1, 2)) / attn.sum(axis=(1, 2))).mean())

    uniform_baseline = {name: MODALITY_NTOKENS[name] / N_TOKENS for name in MODALITY_SLICES}

    print("Attention fraction (mean over all demo samples):")
    print(f"  {'Modality':10s}  {'Actual':>8s}  {'Uniform':>8s}  {'Ratio':>7s}")
    for name in MODALITY_SLICES:
        actual  = modality_frac[name]
        uniform = uniform_baseline[name]
        print(f"  {name:10s}  {actual:8.3f}  {uniform:8.3f}  {actual/uniform:7.2f}×")

    # Aggregate bar chart
    fig, ax = plt.subplots(figsize=(6, 3))
    names   = list(modality_frac.keys())
    actuals = [modality_frac[n] for n in names]
    unifs   = [uniform_baseline[n] for n in names]
    x = np.arange(len(names))
    w = 0.35
    ax.bar(x - w/2, actuals, w, color=[COLORS[n] for n in names], label="Actual", edgecolor="white")
    ax.bar(x + w/2, unifs,   w, color="lightgray", label="Uniform baseline", edgecolor="white")
    ax.set_xticks(x); ax.set_xticklabels(names)
    ax.set_ylabel("Fraction of total attention")
    ax.set_title("Modality attention fraction vs uniform baseline", fontsize=10)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("Attention weights not captured — using ablation-derived fallback.")
    modality_frac = {"tabular": 0.435, "image": 0.105, "text": 0.384, "kg": 0.076}
    attn = None
    scores = predictions["predicted_score"].values


In [ ]:
# Per-prediction attention fraction for 4 representative samples
y_true   = data["y"]
preds_np = scores

tp_mask = (preds_np > 0.52) & (y_true == 1)
tn_mask = (preds_np < 0.48) & (y_true == 0)
fp_mask = (preds_np > 0.52) & (y_true == 0)
fn_mask = (preds_np < 0.48) & (y_true == 1)

def first_or_fallback(mask, fallback_idx):
    idxs = np.where(mask)[0]
    return int(idxs[0]) if len(idxs) > 0 else fallback_idx

EXAMPLES = {
    "TP (predicted+, true+)": first_or_fallback(tp_mask, 0),
    "TN (predicted−, true−)": first_or_fallback(tn_mask, 1),
    "FP (predicted+, true−)": first_or_fallback(fp_mask, 2),
    "FN (predicted−, true+)": first_or_fallback(fn_mask, 3),
}

fig, axes = plt.subplots(2, 2, figsize=(9, 5))
axes = axes.flatten()

for ax, (label, idx) in zip(axes, EXAMPLES.items()):
    if attn is not None:
        sample_total = attn[idx].sum()
        sample_frac  = {name: float(attn[idx, :, sl].sum() / sample_total)
                        for name, sl in MODALITY_SLICES.items()}
    else:
        sample_frac = modality_frac.copy()

    names     = list(sample_frac.keys())
    fractions = [sample_frac[n] for n in names]
    uniforms  = [MODALITY_NTOKENS[n] / N_TOKENS for n in names]

    x = np.arange(len(names))
    w = 0.35
    ax.bar(x - w/2, fractions, w, color=[COLORS[n] for n in names], edgecolor="white", label="Actual")
    ax.bar(x + w/2, uniforms,  w, color="lightgray", edgecolor="white", label="Uniform")
    ax.set_xticks(x); ax.set_xticklabels(names, fontsize=7)
    ax.set_ylabel("Attn fraction", fontsize=7)
    ax.set_title(
        f"{label}\nscore={preds_np[idx]:.3f}, true={'out' if y_true[idx]==1 else 'under'}",
        fontsize=8
    )
    if label.startswith("TP"):
        ax.legend(fontsize=7)

plt.suptitle("Per-prediction modality attention fraction (grey = uniform baseline)", fontsize=10)
plt.tight_layout()
plt.show()


**Interpretation**: Grey bars show the uniform baseline (tabular ≈ 87% by token count; image/text/kg ≈ 4.3% each). Coloured bars show actual attention fractions. The model concentrates disproportionate attention on the text and image single-tokens relative to their size — consistent with the ablation finding that these modalities carry the most signal.

Caveats: (1) With model_dim=16 and 4 heads of 4 dimensions each, attention patterns are noisy and should be read as suggestive, not definitive. (2) Mean pooling treats all 23 token representations equally in the final vector — attention weights describe information routing, not the prediction weight of each modality directly. The ablation deltas in the Results section are a more reliable modality-contribution estimate.

## Limitations

- **Small training universe**: 6 stocks, 1 year of data, 1,242 samples. Modality contribution deltas are single-seed estimates with wide uncertainty bounds; multi-seed evaluation would be needed for confidence intervals.
- **Regime-shift sensitivity**: The model learns from a training period and degrades when market volatility changes structurally (fold 2 val AUC 0.448 vs fold 1's 0.591). This is the primary driver of the negative backtest result, not a data quality issue.
- **All-four underperforms best-single**: On a dataset of this size, adding text and KG alongside image introduces noise rather than signal. More data, not more modalities, is the most direct path to improvement.
- **KG is shallow**: The 4-dimensional KG token encodes only peer count, average peer/sector returns, and a volume event flag. A richer relational feature set (sector rank, peer correlations, regime indicators) would carry more information, but requires a larger peer universe to be meaningful.

## Further reading

- **[README](../../README.md)**: Headline results, architecture summary, and reproduction instructions.
- **[docs/findings.md](../../docs/findings.md)**: Full per-fold diagnostic narrative, modality independence tables, and backtest correction story.
- **[docs/design-notes.md](../../docs/design-notes.md)**: Rationale for GAF/MTF over candlestick ViT, purged CV, and FinBERT integration.
- **[notebooks/colab/run_experiment.ipynb](run_experiment.ipynb)**: Unattended experiment runner for full multi-stock, multi-epoch runs with ablations and backtest. Use this to reproduce or extend Run C.